In [0]:

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def standardize_market_price_columns(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("region", F.upper(F.trim(F.col("region"))))
        .withColumn("market_type", F.upper(F.trim(F.col("market_type"))))
    )

def cast_market_price_fields(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("price_eur_mwh", F.col("price_eur_mwh").cast("double"))
        .withColumn("volume_mwh", F.col("volume_mwh").cast("double"))
    )

def add_market_price_day(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("report_day", F.to_date("event_date"))
    )

def filter_invalid_market_prices(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(F.col("price_eur_mwh").isNotNull())
        .filter(F.col("volume_mwh").isNotNull())
        .filter(F.col("report_day").isNotNull())
    )
def transform_market_prices(df: DataFrame) -> DataFrame:
    """Complete market price transformation pipeline"""
    return (
        df
        .transform(standardize_market_price_columns)
        .transform(cast_market_price_fields)
        .transform(add_market_price_day)
        .transform(filter_invalid_market_prices)
    )
       

In [0]:

import yaml

config_path = "/Workspace/Repos/adb-emily@startsteps.org/vattenfall-week9-capstone-EmilyImunde/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
bronze_schema = config["schemas"]["raw"]
silver_schema = config["schemas"]["refined"]

print(f"Catalog: {catalog_name}")
print(f"Bronze Schema: {bronze_schema}")
print(f"Silver Schema: {silver_schema}")


In [0]:
bronze_df = spark.table(f"{catalog_name}.{bronze_schema}.bronze_market_prices")
silver_df = transform_market_prices(bronze_df)
silver_df.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{silver_schema}.silver_market_prices"
)

print(f"✓ Successfully created {catalog_name}.{silver_schema}.silver_market_prices")
print(f"Total rows: {silver_df.count()}")

In [0]:

silver_table = spark.table("vattenfall_dev.refined.silver_market_prices")

print(f"Total rows: {silver_table.count()}")
print("\nSchema:")
silver_table.printSchema()
print("\nFirst 10 rows:")
display(silver_table.limit(10))

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM vattenfall_dev.refined.silver_market_prices;